# Procurement Intelligence — ML Models
**Microsoft Capstone Project**

This notebook runs all three ML models on EU TED procurement data filtered to IT categories (CPV 48xxx Software, 72xxx IT Services).

| Model | Type | Question for Microsoft |
|-------|------|------------------------|
| 1. Anomaly Detection | Unsupervised (Isolation Forest) | Which IT tenders are unusually priced? |
| 2. Buyer Segmentation | Unsupervised (K-Means) | Which buyers are worth targeting? |
| 3. Competition Intensity | Supervised (XGBoost) | How competitive will this tender be? |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Load IT procurement data
df = spark.table('capstone.ted.silver_lots_enriched')
df = df.filter("cpv_division IN ('48', '72')").toPandas()
df['value_eur'] = df['lot_value_eur'].fillna(df['estimated_value'])
df['issue_date'] = pd.to_datetime(df['issue_date'], errors='coerce')

print(f'Total IT lots: {len(df):,}')
print(f'Countries: {df["buyer_country_code"].nunique()}')
print(f'Buyers: {df["buyer_org_ref"].nunique():,}')
print(f'Date range: {df["issue_date"].min().date()} to {df["issue_date"].max().date()}')

## Model 1 — Contract Value Anomaly Detection
Flags IT tenders where the contract value is unusually high or low compared to similar contracts in the same CPV category and country. Uses Isolation Forest — an unsupervised ML algorithm that learns what a normal contract looks like and scores deviations.

In [ ]:
# Prepare features
df_val = df.dropna(subset=['value_eur'])
df_val = df_val[df_val['value_eur'] > 0].copy()

# Market statistics per country + CPV
market_stats = df_val.groupby(['buyer_country_code', 'cpv_division'])['value_eur'].agg(
    market_median='median', market_mean='mean', market_std='std'
).reset_index()
df_val = df_val.merge(market_stats, on=['buyer_country_code', 'cpv_division'], how='left')
df_val['value_to_median_ratio'] = (df_val['value_eur'] / df_val['market_median'].replace(0, np.nan)).fillna(1).clip(0, 100)
df_val['value_zscore'] = ((df_val['value_eur'] - df_val['market_median']) / df_val['market_std'].replace(0, np.nan)).fillna(0)

df_val['log_value'] = np.log1p(df_val['value_eur'])
df_val['cpv_enc'] = pd.Categorical(df_val['cpv_division']).codes
df_val['country_enc'] = pd.Categorical(df_val['buyer_country_code']).codes
df_val['procedure_enc'] = pd.Categorical(df_val['procurement_procedure'].fillna('UNKNOWN')).codes
df_val['buyer_type_enc'] = pd.Categorical(df_val['buyer_legal_type'].fillna('UNKNOWN')).codes

feature_cols = ['log_value', 'cpv_enc', 'country_enc', 'procedure_enc', 'buyer_type_enc', 'value_zscore', 'value_to_median_ratio']
X = df_val[feature_cols].fillna(0).values
X_scaled = StandardScaler().fit_transform(X)

# Train Isolation Forest
model_if = IsolationForest(contamination=0.05, random_state=42, n_estimators=100)
df_val['is_anomaly'] = model_if.fit_predict(X_scaled) == -1
df_val['anomaly_direction'] = np.where(
    ~df_val['is_anomaly'], 'Normal',
    np.where(df_val['value_eur'] > df_val['market_median'], 'Unusually High', 'Unusually Low')
)

print(f'Lots with known value: {len(df_val):,}')
print(f'Anomalies detected: {df_val["is_anomaly"].sum():,} ({df_val["is_anomaly"].mean()*100:.1f}%)')
print(f'\nAnomaly direction breakdown:')
print(df_val['anomaly_direction'].value_counts())

In [ ]:
# Visualise anomalies
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Value distribution — normal vs anomaly
axes[0].hist(np.log1p(df_val[~df_val['is_anomaly']]['value_eur']), bins=50, alpha=0.6, label='Normal', color='steelblue')
axes[0].hist(np.log1p(df_val[df_val['is_anomaly']]['value_eur']), bins=50, alpha=0.6, label='Anomaly', color='crimson')
axes[0].set_xlabel('Log Contract Value (EUR)')
axes[0].set_ylabel('Count')
axes[0].set_title('Contract Value Distribution — Normal vs Anomaly')
axes[0].legend()

# Plot 2: Anomalies by country
anomaly_by_country = df_val[df_val['is_anomaly']]['buyer_country_code'].value_counts().head(10)
axes[1].barh(anomaly_by_country.index, anomaly_by_country.values, color='crimson')
axes[1].set_xlabel('Number of Anomalies')
axes[1].set_title('Top 10 Countries by Anomalous IT Contracts')

plt.tight_layout()
plt.show()

print('\nTop 10 anomalous IT contracts (highest value):')
display(df_val[df_val['is_anomaly']].sort_values('value_eur', ascending=False)[
    ['buyer_country_code', 'cpv_division', 'lot_name', 'value_eur', 'market_median', 'value_to_median_ratio', 'anomaly_direction']
].head(10))

## Model 2 — IT Buyer Segmentation
Clusters 4,877 public IT buyers into behavioral segments based on their procurement history. Helps Microsoft identify which buyers to target and how to approach them.

In [ ]:
# Build buyer profiles
profiles = df.groupby(['buyer_org_ref', 'buyer_name', 'buyer_country_code']).agg(
    total_tenders=('notice_publication_id', 'nunique'),
    total_value_eur=('value_eur', 'sum'),
    avg_lot_value=('value_eur', 'mean'),
    cpv_diversity=('cpv_code', 'nunique'),
    open_procedure_pct=('procurement_procedure', lambda x: (x.str.lower() == 'open').mean()),
    first_tender=('issue_date', 'min'),
    last_tender=('issue_date', 'max'),
).reset_index()

profiles['active_days'] = (profiles['last_tender'] - profiles['first_tender']).dt.days.clip(lower=1)
profiles['tenders_per_month'] = (profiles['total_tenders'] / (profiles['active_days'] / 30)).clip(upper=50)
profiles = profiles[profiles['total_tenders'] >= 2].copy()

profiles['log_total_value'] = np.log1p(profiles['total_value_eur'].fillna(0))
profiles['log_avg_lot_value'] = np.log1p(profiles['avg_lot_value'].fillna(0))
profiles['log_total_tenders'] = np.log1p(profiles['total_tenders'])

feature_cols = ['log_total_value', 'log_avg_lot_value', 'log_total_tenders', 'tenders_per_month', 'cpv_diversity', 'open_procedure_pct']
X = profiles[feature_cols].fillna(0).values
X_scaled = StandardScaler().fit_transform(X)

# Select optimal k
best_k, best_score = 3, -1
for k in range(3, 7):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    print(f'k={k}: silhouette={score:.3f}')
    if score > best_score:
        best_score, best_k = score, k

print(f'\nOptimal k={best_k} (silhouette={best_score:.3f})')

In [ ]:
# Train final model
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
profiles['segment_id'] = km.fit_predict(X_scaled)

# Assign persona labels
cluster_stats = profiles.groupby('segment_id').agg(
    total_value_eur=('total_value_eur', 'mean'),
    tenders_per_month=('tenders_per_month', 'mean'),
    cpv_diversity=('cpv_diversity', 'mean'),
)
ranked_value = cluster_stats['total_value_eur'].rank(ascending=False)
ranked_freq = cluster_stats['tenders_per_month'].rank(ascending=False)

personas = {}
for cid in cluster_stats.index:
    if ranked_value[cid] == 1 and ranked_freq[cid] <= 2:
        personas[cid] = 'Strategic Enterprise Buyer'
    elif ranked_freq[cid] == 1:
        personas[cid] = 'High-Frequency Buyer'
    elif ranked_value[cid] <= 2:
        personas[cid] = 'High-Value Buyer'
    elif cluster_stats.loc[cid, 'cpv_diversity'] >= cluster_stats['cpv_diversity'].median():
        personas[cid] = 'Specialist IT Buyer'
    else:
        personas[cid] = 'Small Occasional Buyer'

profiles['segment_label'] = profiles['segment_id'].map(personas)

summary = profiles.groupby(['segment_id', 'segment_label']).agg(
    buyer_count=('buyer_org_ref', 'count'),
    avg_tenders=('total_tenders', 'mean'),
    avg_value_eur=('total_value_eur', 'mean'),
    avg_cpv_diversity=('cpv_diversity', 'mean'),
    avg_tenders_per_month=('tenders_per_month', 'mean'),
    top_country=('buyer_country_code', lambda x: x.value_counts().index[0]),
).reset_index()

print('Buyer Segment Summary:')
display(summary)

In [ ]:
# Visualise segments
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Segment sizes
seg_counts = profiles['segment_label'].value_counts()
axes[0].bar(seg_counts.index, seg_counts.values, color=sns.color_palette('tab10', len(seg_counts)))
axes[0].set_xlabel('Segment')
axes[0].set_ylabel('Number of Buyers')
axes[0].set_title('IT Buyer Segments — Size')
axes[0].tick_params(axis='x', rotation=20)

# Plot 2: Avg value per segment
seg_value = profiles.groupby('segment_label')['total_value_eur'].mean().sort_values(ascending=False)
axes[1].barh(seg_value.index, np.log1p(seg_value.values), color=sns.color_palette('tab10', len(seg_value)))
axes[1].set_xlabel('Log Avg Total IT Spend (EUR)')
axes[1].set_title('IT Buyer Segments — Avg Spend')

plt.tight_layout()
plt.show()

## Model 3 — Competition Intensity Predictor
Predicts whether a tender will attract low, medium, or high competition before the deadline closes. Trained on historical award notices where the actual number of submitted bids is known.

> ⏳ **Note:** This model requires `nb_tenders_received` in the Silver layer. Run once the pipeline has been updated.

In [ ]:
# Check if nb_tenders_received is available
cols = spark.table('capstone.ted.silver_lots_enriched').columns
if 'nb_tenders_received' in cols:
    print('nb_tenders_received is available — run competition_intensity.py')
else:
    print('nb_tenders_received not yet available in Silver layer.')
    print('Ask teammate to re-run the bronze pipeline with the updated parser.')

## Summary — Microsoft Intelligence Output
Together the three models answer the key questions for Microsoft's public sector sales team.

In [ ]:
# Combine anomaly flags with buyer segments into a single intelligence view
intel = df_val[['notice_publication_id', 'lot_id', 'lot_name', 'buyer_org_ref',
                'buyer_country_code', 'cpv_division', 'value_eur',
                'is_anomaly', 'anomaly_direction', 'market_median', 'value_to_median_ratio']].copy()

intel = intel.merge(
    profiles[['buyer_org_ref', 'segment_label', 'total_tenders', 'total_value_eur']],
    on='buyer_org_ref', how='left'
)

print('Top Microsoft opportunities today (anomalous value + high-value buyers):')
top = intel[
    (intel['is_anomaly']) &
    (intel['segment_label'].isin(['Strategic Enterprise Buyer', 'High-Value Buyer']))
].sort_values('value_eur', ascending=False)

display(top[['buyer_country_code', 'cpv_division', 'lot_name', 'value_eur',
             'anomaly_direction', 'segment_label']].head(10))